In [ ]:
# USAGE EXAMPLE: Full preprocessing pipeline for BEL EEG System One (280-channel)
# Works with EGI .mff, FIF, or any MNE-supported format

import mne
from pathlib import Path
from xeeg_kit import execute_meegkit, execute_icalabel, bel_280

# ────────────────────────────────────────────────
# 1. LOAD DATA (any format)
# ────────────────────────────────────────────────
raw = mne.io.read_raw_egi("subject.mff", preload=True)   # EGI .mff
# raw = mne.io.read_raw_fif("subject.fif", preload=True) # FIF

# ────────────────────────────────────────────────
# 2. STANDARDIZE FOR BEL 280-CHANNEL SYSTEM (optional but recommended)
# ────────────────────────────────────────────────
# Example: EGI exports channels as '1', '2', ..., '257' + reference
rename_map = {str(i): f'E{i}' for i in range(1, 281)}
rename_map['REF CZ'] = 'Cz'  # Map reference to Cz

# Apply renaming
raw.rename_channels(rename_map)

# Apply montage from BEL .gpsc file
channels = bel_280.parse_gpsc("ghw280_from_egig.gpsc")
montage = bel_280.create_montage_from_gpsc(channels)
raw.set_montage(montage, on_missing="warn")

# ────────────────────────────────────────────────
# 3. DEFINE CLEANING PARAMETERS
# ────────────────────────────────────────────────
meegkit_params = {
    'low_pass_filter': 100.0,        # High-frequency cutoff (Hz)
    'notch_filter_freq': 60.0,       # Line noise frequency (set to None to skip)
    'mad_threshold': 4.5,            # Bad channel detection sensitivity
    'asr_cutoff': 3.0,               # ASR artifact rejection threshold
    'star_thresh': 2.0,              # STAR denoising threshold
    'sns_neighbors': 8,              # SNS spatial neighbors
    'drop_cz': True,                 # Remove Cz before processing
    'interpolate_bads': True,        # Interpolate bad channels at the end
    'find_cleanest_segment_start': 0.0,
    'find_cleanest_segment_duration': 30.0,
    'verbose': True
}

icalabel_params = {
    'mad_threshold': 10.0,           # Bad channel detection for ICA
    'min_amplitude_uv': 0.1,         # Minimum amplitude to avoid flat channels
    'n_components': 0.98,            # ICA: retain 98% of variance
    'random_state': 97,              # For reproducibility
    'interpolate_bads': True,        # Interpolate original bads after ICA
    'verbose': True
}

# ────────────────────────────────────────────────
# 4. RUN CLEANING PIPELINE
# ────────────────────────────────────────────────
clean_1 = execute_meegkit(raw, **meegkit_params)
clean_2 = execute_icalabel(clean_1, **icalabel_params)

# ────────────────────────────────────────────────
# 5. SAVE FINAL RESULT
# ────────────────────────────────────────────────
output_path = Path("final_cleaned.fif")
output_path.parent.mkdir(parents=True, exist_ok=True)
clean_2.save(output_path, overwrite=True)

print(f"✅ Preprocessing complete. Saved to: {output_path}")